In [ ]:
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm tensorboard

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    
    # Google Driveをマウント
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリに移動
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    
    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from src.training import TrainingPipeline
from src.training.config import TrainingConfig, ModelConfig, DatasetConfig, OptimizerConfig, TrainingPipelineConfig

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
data_root = 'data/detect'

train_dirs = [
    f"{data_root}/01_train",
    f"{data_root}/09_train",
    f"{data_root}/10_train",
    f"{data_root}/11_train",
    f"{data_root}/13_train",
    f"{data_root}/02_train",
    f"{data_root}/03_train",
    f"{data_root}/04_train",
    f"{data_root}/06_train",
]

val_dirs = [
    f"{data_root}/05_train",
    f"{data_root}/07_train",
    f"{data_root}/08_train",
    f"{data_root}/12_train",
    f"{data_root}/14_train",
]

OUTPUT_DIR = "output/training"

print("=" * 60)
print("データディレクトリの確認")
print("=" * 60)
import os
print("訓練用データ:")
for i, data_dir in enumerate(train_dirs, 1):
    exists = os.path.exists(data_dir)
    csv_exists = os.path.exists(os.path.join(data_dir, 'player_pose_data.csv'))
    label_exists = os.path.exists(os.path.join(data_dir, 'play_labels.csv'))
    print(f"  [{i}] {os.path.basename(data_dir)}: {'✓' if exists else '✗'}")
    if exists:
        print(f"      - player_pose_data.csv: {'✓' if csv_exists else '✗'}")
        print(f"      - play_labels.csv: {'✓' if label_exists else '✗'}")
print(f"\n検証用データ:")
for i, data_dir in enumerate(val_dirs, 1):
    exists = os.path.exists(data_dir)
    csv_exists = os.path.exists(os.path.join(data_dir, 'player_pose_data.csv'))
    label_exists = os.path.exists(os.path.join(data_dir, 'play_labels.csv'))
    print(f"  [{i}] {os.path.basename(data_dir)}: {'✓' if exists else '✗'}")
    if exists:
        print(f"      - player_pose_data.csv: {'✓' if csv_exists else '✗'}")
        print(f"      - play_labels.csv: {'✓' if label_exists else '✗'}")
print("=" * 60)
print(f"\n訓練: {len(train_dirs)}動画 / 検証: {len(val_dirs)}動画")
print("各ディレクトリに player_pose_data.csv と play_labels.csv が必要です")

In [ ]:
# ========================================
# 設定ファイルの読み込み
# ========================================
with open("configs/training_config.json", "r") as f:
    config_dict = json.load(f)

# デバイス設定（GPU/CPU自動判定）
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 各設定を作成
model_config = ModelConfig(**config_dict['model'])

dataset_config = DatasetConfig(
    train_data_dirs=train_dirs,
    val_data_dirs=val_dirs,
    **config_dict['dataset']
)

optimizer_config = OptimizerConfig(**config_dict['optimizer'])

training_config = TrainingConfig(
    **{**config_dict['training'], 'device': device}
)

# パイプライン設定を作成
pipeline_config = TrainingPipelineConfig(
    model=model_config,
    dataset=dataset_config,
    optimizer=optimizer_config,
    training=training_config,
    output_dir=config_dict.get('output_dir', 'output/training')
)

# パイプライン作成
pipeline = TrainingPipeline(pipeline_config)

print("設定ファイル: configs/training_config.json")
print(f"デバイス: {device}")

In [ ]:
# ========================================
# 学習実行
# ========================================

# パイプラインを実行（全自動）
results = pipeline.run()

print("\n学習完了！")
print(f"  Best F1 Score: {results['best_val_f1']:.4f}")
print(f"  Best Model: {results['best_model_path']}")
print(f"  Final Model: {results['final_model_path']}")
print(f"  Training History: {results['history_path']}")

# 結果を保存（後で参照用）
output_dir = Path(results['output_dir'])
timestamp = output_dir.name

In [ ]:
# ========================================
# 学習曲線の可視化
# ========================================

# 学習履歴の読み込み
history_path = output_dir / 'training_history.json'
with open(history_path, 'r') as f:
    history = json.load(f)

# プロット
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
if history['val_acc']:
    axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
if history['val_f1']:
    axes[2].plot(history['val_f1'], label='Val', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('F1 Score', fontsize=12)
axes[2].set_title('Training and Validation F1 Score', fontsize=14)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n学習曲線を保存: {output_dir / 'training_curves.png'}")

In [ ]:
# ========================================
# モデルの評価（オプション）
# ========================================

from src.models.play_classifier_lstm import PlayClassifierLSTM

# ベストモデルを読み込み
best_model_path = output_dir / 'best_model.pth'
checkpoint = torch.load(best_model_path, map_location=device)

# 保存された設定からモデルを再作成
saved_config_path = output_dir / 'config.json'
with open(saved_config_path, 'r') as f:
    saved_config = json.load(f)

model = PlayClassifierLSTM(
    input_size=102,  # 座標34 + 速度34 + 加速度34
    hidden_size=saved_config['model']['hidden_size'],
    num_layers=saved_config['model']['num_layers'],
    dropout=saved_config['model']['dropout'],
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"ベストモデルを読み込み: {best_model_path}")
print(f"  エポック: {checkpoint['epoch']}")
print(f"  Best Val F1: {checkpoint['best_val_f1']:.4f}")

print("\n評価はpredict用のノートブックで実行してください")

In [ ]:
# ========================================
# Google Driveに保存（Colab環境の場合）
# ========================================

import shutil

if IN_COLAB:
    SAVE_TO_DRIVE = f"/content/drive/MyDrive/trained_models/play_classifier/{timestamp}"
    
    os.makedirs(SAVE_TO_DRIVE, exist_ok=True)
    
    files_to_copy = [
        'best_model.pth',
        'final_model.pth',
        'config.json',
        'training_history.json',
        'training_curves.png'
    ]
    
    for filename in files_to_copy:
        src = output_dir / filename
        if src.exists():
            dst = os.path.join(SAVE_TO_DRIVE, filename)
            shutil.copy2(src, dst)
            print(f"コピー完了: {filename} -> {dst}")
    
    print(f"\nモデルをGoogle Driveに保存: {SAVE_TO_DRIVE}")
else:
    print("ローカル環境では自動保存はスキップされます")
    print(f"出力ディレクトリ: {output_dir}")